# 微小卫星遥测智能异常检测与RAG解释辅助系统
## 完整Pipeline演示Notebook（段级18维特征版）

**项目**: 上海市大学生创新创业训练计划  
**团队**: 上海电机学院 电子信息学院  
**数据集**: ESA OPS-SAT-AD (2123段, 9通道, 20.4%异常率［段级口径：434/2123］)  
**核心方法**: Isolation Forest + 规则兜底 + RAG智能解释  
**特征**: 段级18维统计特征（完全对齐预研报告实验设定）

---

---
## 0. 环境初始化

导入 Python 核心库（numpy、pandas、sklearn、plotly），设置项目根目录路径，确保后续模块可正确加载。

In [ ]:
import sys, os
from pathlib import Path
import json, time
from datetime import datetime

PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT") or Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import warnings
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score, precision_score, recall_score, matthews_corrcoef, confusion_matrix
import plotly.graph_objects as go

warnings.filterwarnings('ignore')
print("✅ 环境设置完成")
print(f"项目根目录: {PROJECT_ROOT}")

---
## 1. 数据加载与特征提取

加载从 `segments.csv`（原始遥测，303K 行 × 8 列）提取的段级 18 维统计特征文件 `segments_18d.csv`（2123 段 × 23 列），特征: 段级18维统计特征（完全对齐预研报告实验设定），包括：基础统计7维（duration, len, mean, var, std, kurtosis, skew）、峰值检测5维（n_peaks, smooth10/20_n_peaks, diff/diff2_peaks）、差分方差2维（diff_var, diff2_var）、衍生特征4维（gaps_squared, len_weighted, var_div_duration, var_div_len）

In [ ]:
print("=== 加载段级18维特征 ===")

# 加载 segments_18d.csv
segments_18d_path = PROJECT_ROOT / "data" / "processed" / "segments_18d.csv"

if segments_18d_path.exists():
    features_df = pd.read_csv(segments_18d_path)
    print("✅ 已加载预生成的18维特征文件")
else:
    print("⚠️ 未找到预生成文件，正在从原始数据提取...")
    from features.extract_segments_18d import generate_dataset
    
    raw_path = PROJECT_ROOT / "data" / "raw" / "segments.csv"
    segments_raw = pd.read_csv(raw_path, parse_dates=["timestamp"])
    
    features_df = generate_dataset(segments_raw)
    
    os.makedirs(segments_18d_path.parent, exist_ok=True)
    features_df.to_csv(segments_18d_path, index=False, encoding="utf-8-sig")
    print(f"✅ 已生成并保存: {segments_18d_path}")

print(f"\n=== 数据集概览 ===")
print(f"总段数: {len(features_df)} | 特征维度: {features_df.shape[1]}列")
print(f"通道数: {features_df['channel'].nunique()} | 异常率: {features_df['anomaly'].mean()*100:.1f}%")
print(f"训练段: {features_df['train'].sum()} | 测试段: {(~features_df['train'].astype(bool)).sum()}")

# BUG #2 FIX: META_COLS 去掉 duration，确保18维特征
META_COLS = ['segment', 'anomaly', 'train', 'channel', 'sampling']
feature_cols = [c for c in features_df.columns if c not in META_COLS]

print(f"\n元数据({len(META_COLS)}): {META_COLS}")
print(f"特征列({len(feature_cols)}): {feature_cols}")
print(features_df.head())

---
## 2. 训练/测试集划分

使用数据集预设的 `train` 列划分训练集（1594 段）和测试集（529 段），与 OPS-SAT-AD 原论文基准完全对齐。

In [ ]:
# 使用数据集预设的train列划分（与论文基准一致）
# BUG #1 FIX: reset_index 避免 numpy 数组索引越界
train_df = features_df[features_df['train'] == True].copy().reset_index(drop=True)
test_df = features_df[features_df['train'] == False].copy().reset_index(drop=True)

print("=== 训练/测试集划分 ===")
print(f"训练段数: {len(train_df)} | 测试段数: {len(test_df)}")
print(f"训练异常率: {train_df['anomaly'].mean()*100:.1f}% | 测试异常率: {test_df['anomaly'].mean()*100:.1f}%")

---
## 3. 工具函数定义

定义后续各 Stage 共用的核心函数：
- `eval_iforest_segment()`: 训练 IForest 并返回段级预测
- `calc_segment_metrics()`: 计算 F1/Precision/Recall/MCC
- `compute_stat_thresholds()`: **分通道**计算 3σ 和 IQR 阈值（与预研报告一致）
- `apply_rules()`: 对测试数据应用**分通道**统计规则（违反>=2条特征即判异常）


In [ ]:
def eval_iforest_segment(X_train, X_test, contamination=0.2, random_state=42):
    """训练IForest并返回段级预测"""
    clf = IsolationForest(n_estimators=100, contamination=contamination, random_state=random_state, n_jobs=-1)
    clf.fit(X_train)
    return np.where(clf.predict(X_test) == -1, 1, 0), clf

def calc_segment_metrics(y_true, y_pred):
    """计算段级完整指标"""
    return {
        'f1': float(f1_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'mcc': float(matthews_corrcoef(y_true, y_pred))
    }

def compute_stat_thresholds(train_df, channels, feature_cols):
    """计算各通道独立的3σ和IQR统计阈值（与预研报告一致）"""
    thresholds = {}
    for ch in channels:
        ch_data = train_df[train_df['channel'] == ch]
        ch_th = {}
        for col in feature_cols:
            vals = ch_data[col].dropna()
            if len(vals) < 5:
                continue
            std = float(vals.std())
            ch_th[col] = {
                'mean': float(vals.mean()),
                'std': std if std > 1e-6 else 1.0,
                'q1': float(vals.quantile(0.25)),
                'q3': float(vals.quantile(0.75)),
                'iqr': float(vals.quantile(0.75) - vals.quantile(0.25)),
            }
        thresholds[ch] = ch_th
    return thresholds

def apply_rules(test_df, thresholds, feature_cols):
    """应用分通道统计规则，返回二值预测（违反>=2条即为异常）"""
    rule_pred = np.zeros(len(test_df))
    for i, (idx, row) in enumerate(test_df.iterrows()):
        ch = row['channel']
        if ch not in thresholds:
            continue
        ch_th = thresholds[ch]
        count = 0
        for col in feature_cols:
            if col not in ch_th or pd.isna(row[col]):
                continue
            v = row[col]
            th = ch_th[col]
            if abs(v - th['mean']) > 3.0 * th['std']:
                count += 1
            lo = th['q1'] - 1.5 * th['iqr']
            hi = th['q3'] + 1.5 * th['iqr']
            if v < lo or v > hi:
                count += 1
        rule_pred[i] = 1 if count >= 2 else 0
    return rule_pred

print("✅ 工具函数定义完成")


---
## 4. Stage 0: 全局 IForest Baseline

使用全部 9 个通道的 18 维特征训练一个全局 Isolation Forest（`c=0.2`，与 OPS-SAT-AD 原论文设定一致），作为后续改进的基准线。

In [ ]:
X_train = train_df[feature_cols].values
X_test = test_df[feature_cols].values
y_test_true = test_df['anomaly'].values

CONTAMINATION = 0.2
RANDOM_STATE = 42

print("=== Stage 0: 全局 IForest Baseline ===")
print(f"特征维度: {len(feature_cols)} | 训练段: {len(X_train)} | 测试段: {len(X_test)}")

y_pred_stage0, clf_stage0 = eval_iforest_segment(X_train, X_test, CONTAMINATION, RANDOM_STATE)
metrics_stage0 = calc_segment_metrics(y_test_true, y_pred_stage0)

print("\n段级指标 (Stage 0):")
for k, v in metrics_stage0.items():
    print(f"  {k}: {v:.4f}")

cm0 = confusion_matrix(y_test_true, y_pred_stage0)
print(f"  混淆矩阵: TN={cm0[0][0]}, FP={cm0[0][1]}, FN={cm0[1][0]}, TP={cm0[1][1]}")

---
## 5. Stage 1: 分通道独立 IForest

**创新点 2 — 通道级自适应建模**。9 个遥测通道的数据分布差异显著（强通道 vs 弱通道），全局模型难以兼顾。本节为每个通道独立训练 IForest，使每个子模型专注于自身通道的分布特征。

In [ ]:
STRONG_CHANNELS = ['CADC0872', 'CADC0873', 'CADC0874']
WEAK_CHANNELS = ['CADC0884', 'CADC0886', 'CADC0888', 'CADC0890', 'CADC0892', 'CADC0894']
ALL_CHANNELS = STRONG_CHANNELS + WEAK_CHANNELS

print("=== Stage 1: 分通道独立 IForest ===")
print(f"强通道: {STRONG_CHANNELS}")
print(f"弱通道: {WEAK_CHANNELS}")

channel_models = {}
y_pred_stage1 = np.zeros(len(test_df), dtype=int)

for ch in ALL_CHANNELS:
    ch_train = train_df[train_df['channel'] == ch]
    ch_test_idx = test_df[test_df['channel'] == ch].index
    ch_test = test_df[test_df['channel'] == ch]
    
    if len(ch_train) < 10 or len(ch_test) == 0:
        print(f"  {ch}: 样本不足，跳过")
        continue
    
    clf = IsolationForest(n_estimators=100, contamination=CONTAMINATION, random_state=RANDOM_STATE, n_jobs=-1)
    clf.fit(ch_train[feature_cols].values)
    channel_models[ch] = clf
    y_pred_stage1[ch_test_idx] = np.where(clf.predict(ch_test[feature_cols].values) == -1, 1, 0)
    print(f"  {ch}: 训练={len(ch_train)}, 测试={len(ch_test)}")

metrics_stage1 = calc_segment_metrics(y_test_true, y_pred_stage1)

print("\n段级指标 (Stage 1):")
for k, v in metrics_stage1.items():
    print(f"  {k}: {v:.4f}")

imp_s0_s1 = metrics_stage1['f1'] - metrics_stage0['f1']
print(f"\n相比 Stage 0 提升: +{imp_s0_s1:.4f} (+{imp_s0_s1/metrics_stage0['f1']*100:.1f}%)")

---
## 6. Stage 2: IF + 统计规则融合

**创新点 3 — 多策略融合与规则兜底**。在 Stage 1 基础上引入**分通道** 3σ/IQR 统计规则：
- **强通道**（CADC0872/873/874）：IF 预测 OR 规则预测
- **弱通道**（其余 6 个）：仅使用统计规则

关键：统计阈值在**每个通道的训练数据**上独立计算（非全局），与预研报告 `fair_comparison_v2_segments_c02.py` 完全对齐。规则阈值：单通道违反 ≥2 条特征异常即判定为异常段。


In [ ]:
print("=== Stage 2: IF + 规则融合 ===")

# 分通道计算统计阈值（与预研报告一致）
thresholds = compute_stat_thresholds(train_df, ALL_CHANNELS, feature_cols)
print(f"统计阈值计算完成（{len(thresholds)} 个通道独立计算）")

# 分通道规则预测
rule_pred = apply_rules(test_df, thresholds, feature_cols)
print(f"规则预测完成：异常段数 = {int(rule_pred.sum())}")

y_pred_stage2 = np.zeros(len(test_df), dtype=int)
for ch in ALL_CHANNELS:
    ch_test = test_df[test_df['channel'] == ch]
    ch_test_idx = ch_test.index
    if len(ch_test) == 0:
        continue
    
    r_pred = rule_pred[ch_test_idx]
    
    if ch in STRONG_CHANNELS and ch in channel_models:
        # 强通道：IF + 规则 OR
        if_pred = np.where(channel_models[ch].predict(ch_test[feature_cols].values) == -1, 1, 0)
        final = np.maximum(if_pred, r_pred)
        print(f"  {ch}: 强通道 OR融合")
    else:
        # 弱通道：纯规则
        final = r_pred
        print(f"  {ch}: 弱通道 纯规则")
    
    y_pred_stage2[ch_test_idx] = final

metrics_stage2 = calc_segment_metrics(y_test_true, y_pred_stage2)
print("\n段级指标 (Stage 2):")
for k, v in metrics_stage2.items():
    print(f"  {k}: {v:.4f}")

imp_s1_s2 = metrics_stage2['f1'] - metrics_stage1['f1']
total_imp = metrics_stage2['f1'] - metrics_stage0['f1']
print(f"\n总提升 (S0→S2): {metrics_stage0['f1']:.4f} → {metrics_stage2['f1']:.4f} (+{total_imp/metrics_stage0['f1']*100:.1f}%)")


---
## 7. 各阶段性能对比与基准对比

汇总三阶段实验的 SegF1/Precision/Recall/MCC 指标，与 OPS-SAT-AD 论文中的代表性方法（IForest/OCSVM/MO-GAAL/FCNN 等）进行横向对比，量化本文方法的增益。

In [ ]:
results_df = pd.DataFrame({
    'Stage': ['0.全局IF', '1.分通道IF', '2.IF+规则'],
    'SegF1': [metrics_stage0['f1'], metrics_stage1['f1'], metrics_stage2['f1']],
    'Precision': [metrics_stage0['precision'], metrics_stage1['precision'], metrics_stage2['precision']],
    'Recall': [metrics_stage0['recall'], metrics_stage1['recall'], metrics_stage2['recall']],
    'MCC': [metrics_stage0['mcc'], metrics_stage1['mcc'], metrics_stage2['mcc']]
})
results_df['F1提升'] = results_df['SegF1'] - results_df['SegF1'].iloc[0]

print("=== 各阶段性能对比（段级18维特征）===")
print(results_df.to_string(index=False, float_format='%.4f'))

print("\n=== 与 OPS-SAT-AD 基准对比 ===")
baselines = pd.DataFrame([
    {'方法': 'IForest(论文基线)', 'SegF1': 0.295, '备注': '18维,c=0.2'},
    {'方法': '本文全局IF', 'SegF1': metrics_stage0['f1'], '备注': f'{len(feature_cols)}维'},
    {'方法': '本文最终方法', 'SegF1': metrics_stage2['f1'], '备注': 'IF+分通道+规则'},
    {'方法': 'OCSVM(论文)', 'SegF1': 0.647, '备注': '经典单类方法'},
    {'方法': 'MO-GAAL(论文)', 'SegF1': 0.726, '备注': '当前最佳无监督'},
    {'方法': 'FCNN(有监督)', 'SegF1': 0.946, '备注': '有监督上限'}
])
print(baselines.to_string(index=False, float_format='%.3f'))

print(f"\n📊 关键结论:")
print(f"  本文全局IF({metrics_stage0['f1']:.3f}) vs 论文基线(0.295)")
print(f"  最终方法 SegF1={metrics_stage2['f1']:.3f}, 较基线提升 {total_imp/metrics_stage0['f1']*100:.1f}%")
print(f"  大幅缩小与MO-GAAL(0.726)差距，复杂度极低")

---
## 8. 性能提升可视化

使用 Plotly 柱状图展示三阶段递进优化的 SegF1 变化趋势，直观呈现每一步改进的效果。

In [ ]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=results_df['Stage'],
    y=results_df['SegF1'],
    text=[f"{v:.4f}" for v in results_df["SegF1"]],
    textposition='outside',
    marker_color=['#1f77b4', '#ff7f0e', '#2ca02c']
))

fig.update_layout(
    title='异常检测性能提升 (段级 SegF1 - 18维特征)',
    yaxis_title='F1 Score',
    yaxis_range=[0, max(results_df['SegF1'])*1.2],
    template='plotly_white',
    width=700,
    height=500
)

fig.show()

---
## 9. RAG 模块初始化

导入 AnomalyRAGPipeline，加载 BGE-M3 嵌入模型 + FAISS 向量索引（5064 chunks，来源：NASA SOA、ITU 等），为后续异常解释做准备。

In [ ]:
try:
    # BUG #8 FIX: sys.path 已包含 PROJECT_ROOT/src，不能用 src. 前缀
    from integration.anomaly_rag_pipeline import AnomalyRAGPipeline
    # 切换到项目根目录，确保 FAISS 等相对路径正确
    os.chdir(PROJECT_ROOT)
    RAG_AVAILABLE = True
    print("✅ RAG模块可用")
except ImportError as e:
    RAG_AVAILABLE = False
    print("=" * 60)
    print(f"⚠️  RAG不可用（导入失败）: {e}")
    print("⚠️  已跳过RAG解释部分，但前面的异常检测 Pipeline 仍正常可用")
    print("如需启用RAG解释，请确认：")
    print("  1) 已执行 pip install -r requirements.txt（含 faiss-gpu / langchain 等）")
    print("  2) 已将 BGE-M3 本地模型放置到 models/ 目录")
    print("  3) 已在 .env 中配置 VOLCENGINE_API_KEY")
    print("=" * 60)

---
## 10. RAG 智能解释生成

将 Stage 2 检测到的异常段送入 RAG Pipeline，基于 NASA SOA、ITU 等 6 个知识源（5064 chunks）为每个异常生成诊断解释。本节演示前 3 个异常段的解释结果。

In [ ]:
if RAG_AVAILABLE:
    pipeline = AnomalyRAGPipeline()
    
    # 使用 Stage2 预测结果构建异常列表
    anomalies_detected = [
        {
            'segment': int(test_df.loc[i, 'segment']),
            'channel': str(test_df.loc[i, 'channel']),
            'is_true_anomaly': bool(test_df.loc[i, 'anomaly'])
        }
        for i in test_df.index if y_pred_stage2[i] == 1
    ]
    
    print(f"\nStage2 检测到 {len(anomalies_detected)} 个异常段")
    
    # BUG #5 FIX: 将 dict 转为对象（RAG pipeline 接口需要属性访问）
    class AnomalyInfo:
        def __init__(self, d):
            self.segment = d['segment']
            self.channel = d['channel']
            self.is_true_anomaly = d['is_true_anomaly']
    
    rag_results = []
    for i, a in enumerate(anomalies_detected[:3]):
        print(f"\n--- 异常 {i+1} | 段{a['segment']} | {a['channel']} | 真实={a['is_true_anomaly']} ---")
        try:
            a_obj = AnomalyInfo(a)
            rr = pipeline.explain(a_obj)
            # 兼容：answer 可能为空 / None / 普通对象
            if hasattr(rr, "answer") and rr.answer:
                ex = str(rr.answer)[:500]
            elif rr and not hasattr(rr, "answer"):
                ex = str(rr)[:500]
            else:
                ex = ""
            if not ex:
                print("⚠️  该异常段未返回任何解释（模型/索引/key 可能异常）")
                rag_results.append({
                    'rank': i+1,
                    'segment_id': a['segment'],
                    'channel': a['channel'],
                    'explanation': 'Error: 空解释（请检查 BGE-M3 模型 / VOLCENGINE_API_KEY / FAISS 索引）'
                })
            else:
                print(f"RAG解释: {ex}...")
                rag_results.append({
                    'rank': i+1,
                    'segment_id': a['segment'],
                    'channel': a['channel'],
                    'explanation': ex
                })
        except Exception as e:
            print(f"错误: {e}")
            rag_results.append({
                'rank': i+1,
                'segment_id': a['segment'],
                'channel': a['channel'],
                'explanation': f'Error: {str(e)[:200]}'
            })
    
    # ---- 结果校验：避免“静默跑通”假象 ----
    _bad = [r for r in rag_results if not r.get('explanation') or str(r.get('explanation','')).startswith('Error')]
    if len(rag_results) == 0:
        print("=" * 60)
        print("⚠️  RAG 模块已加载，但本次未产出任何解释！")
        print("⚠️  请检查：BGE-M3 模型是否就位 / VOLCENGINE_API_KEY 是否配置 / FAISS 索引是否有效")
        print("=" * 60)
    elif _bad:
        print("⚠️  警告：{}/{} 个异常段的 RAG 解释失败，结果可能不完整".format(len(_bad), len(rag_results)))
    else:
        print(f"✅ RAG 解释生成完成：共 {len(rag_results)} 条")
else:
    rag_results = []
    print("=" * 60)
    print("⚠️  RAG 不可用 —— 已跳过解释生成（前面的异常检测结果仍有效）")
    print("=" * 60)

---
## 11. 结果导出

将全部实验结果（三阶段指标、逐段预测详情、RAG解释、统计阈值、基准对比）导出为结构化 JSON 文件，便于复现与答辩展示。

In [ ]:
output = {
    "metadata": {
        "project": "微小卫星遥测智能异常检测与RAG解释辅助系统",
        "dataset": "ESA OPS-SAT-AD",
        "timestamp": datetime.now().isoformat(),
        "version": "2.0-segment-18d",
        "feature_type": "段级18维统计特征"
    },
    "data_info": {
        "n_segments_total": int(len(features_df)),
        "n_channels": int(features_df['channel'].nunique()),
        "anomaly_rate": float(features_df['anomaly'].mean()),
        "n_train_segments": int(len(train_df)),
        "n_test_segments": int(len(test_df))
    },
    "model_config": {
        "feature_dim": len(feature_cols),
        "feature_cols": feature_cols,
        "contamination_default": CONTAMINATION,
        "random_state": RANDOM_STATE,
        "strong_channels": STRONG_CHANNELS,
        "weak_channels": WEAK_CHANNELS
    },
    "stages_performance": {
        "stage0_global_if": {
            "description": "全局IForest, c=0.2",
            "metrics": metrics_stage0
        },
        "stage1_per_channel": {
            "description": "分通道独立IForest, c=0.2",
            "metrics": metrics_stage1,
            "improvement_over_s0_pct": float((metrics_stage1['f1']-metrics_stage0['f1'])/metrics_stage0['f1']*100)
        },
        "stage2_fusion": {
            "description": "IF + 规则融合（强通道OR，弱通道纯规则）",
            "metrics": metrics_stage2,
            "total_improvement_pct": float(total_imp/metrics_stage0["f1"]*100)
        }
    },
    "predictions_detail": [
        {
            "segment_id": int(test_df.loc[i, 'segment']),
            "channel": str(test_df.loc[i, 'channel']),
            "is_anomaly_true": bool(test_df.loc[i, 'anomaly']),
            "predictions": {
                "s0_global_if": int(y_pred_stage0[i]),
                "s1_per_channel": int(y_pred_stage1[i]),
                "s2_fusion": int(y_pred_stage2[i])
            }
        }
        for i in test_df.index
    ],
    "rag_explanations": rag_results,
    # BUG #7 FIX: 二层遍历，避免对内层 dict 调用 float()
    "stat_thresholds": {
        col: {feat: {kk: float(vv) for kk, vv in vals.items()} for feat, vals in t.items()}
        for col, t in thresholds.items()
    },
    "baseline_comparison": [
        {"method": "IForest(论文基线)", "SegF1": 0.295, "note": "18维,c=0.2"},
        {"method": "本文全局IF", "SegF1": float(metrics_stage0["f1"]), "note": f"{len(feature_cols)}维"},
        {"method": "本文最终方法", "SegF1": float(metrics_stage2["f1"]), "note": "IF+规则"},
        {"method": "OCSVM(论文)", "SegF1": 0.647, "note": "经典单类"},
        {"method": "MO-GAAL(论文)", "SegF1": 0.726, "note": "最佳无监督"},
        {"method": "FCNN(有监督)", "SegF1": 0.946, "note": "有监督上限"}
    ]
}

# ====== 保存文件 ======
out_main = PROJECT_ROOT / "data" / "results" / "pipeline_segment_18d_full.json"
os.makedirs(out_main.parent, exist_ok=True)
with open(out_main, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print(f"✅ 完整结果: {out_main} ({out_main.stat().st_size/1024:.1f} KB)")

out_bak = PROJECT_ROOT / "data" / "results" / "pipeline_segment_18d_output.json"
with open(out_bak, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print(f"✅ 备份: {out_bak}")

summary = {
    "metadata": output["metadata"],
    "best_segF1": float(metrics_stage2["f1"]),
    "total_improvement_pct": float(total_imp/metrics_stage0["f1"]*100),
    "baseline_comparison": output["baseline_comparison"]
}
out_sum = PROJECT_ROOT / "data" / "results" / "pipeline_segment_18d_summary.json"
with open(out_sum, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"✅ 摘要: {out_sum}")

print("\n" + "="*60)
print("📊 JSON 导出完成！包含:")
print("  1. metadata - 项目元信息和版本标识")
print("  2. data_info - 数据集统计（段数/通道数/异常率）")
print("  3. model_config - 模型配置（18维特征列、通道分类）")
print("  4. stages_performance - 三阶段性能指标 + 提升百分比")
print("  5. baseline_comparison - 与论文基准横向对比（6种方法）")
print("  6. predictions_detail - 每个测试段的详细预测结果 (S0/S1/S2)")
print("  7. rag_explanations - RAG 智能解释（前3个异常）")
print("  8. stat_thresholds - 统计规则阈值（3σ/IQR）")
print("="*60)


---

**✅ Notebook 运行完成！**

### 输出文件：
1. `pipeline_segment_18d_full.json` - 完整结果  
2. `pipeline_segment_18d_output.json` - 备份副本  
3. `pipeline_segment_18d_summary.json` - 精简摘要

### 关键特性：
- ✅ **段级检测**：直接使用段级18维特征，模型直接输出段级标签
- ✅ **三阶段递进**：Global IF → 分通道IF → IF+规则融合
- ✅ **JSON结构化导出**：完整实验可复现信息

### 已修复的Bug（8个）：
- ✅ Bug #1: `reset_index(drop=True)` 解决索引越界
- ✅ Bug #2: META_COLS 去掉 `duration`，确保18维特征
- ✅ Bug #3: 结论文字修正（`vs` 替代 `优于`）
- ✅ Bug #4: `best_params['contamination']` 拼写修正
- ✅ Bug #5: RAG路径 + dict→object 接口适配
- ✅ Bug #6: notebook source 格式规范化
- ✅ Bug #7: `stat_thresholds` 序列化修复（嵌套dict二层遍历，避免对dict调用float()崩溃）
- ✅ Bug #8: RAG导入路径修复（`from integration...` 替代 `from src.integration...`，避免ImportError）
